# Training behaviors for a frozen model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pfekin/LARA/blob/main/examples/behaviors/Qwen3-1.7B/train.ipynb)

This notebook trains a set of behaviors on one frozen model and saves each as a
folder of a few megabytes. The companion notebook, `test`, downloads them
and shows what they do when routed together.

A behavior is a low-rank correction applied between transformer blocks. The base
model is never modified, so several behaviors can sit on it at once and each one
carries a strength you set at inference rather than at training time.

Point `BASE` at any causal language model. Nothing here assumes a particular
one, and the same list of behaviors trains against whatever you choose.

**Runtime.** Roughly 15 minutes per behavior on a T4 at 1.7B parameters. Only one
model is resident at a time.

## Configuration

In [1]:
!pip install -q git+https://github.com/pfekin/LARA.git
!pip install -q transformers datasets accelerate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import gc, json, math, os, random
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

from lara import LARA, Bank

NL = chr(10)

# ── the model ────────────────────────────────────────────────────────────────
# Any causal LM. Nothing below assumes a particular one.
BASE = "Qwen/Qwen3-1.7B"

# ── the behaviors ────────────────────────────────────────────────────────────
# Comment entries out to train fewer. Everything adapts to the list.
# `layers` and `rank` override the defaults per behavior: preference training
# needs far less placement than fine-tuning, so `polite` uses a single module.
BEHAVIORS = [
    {"name": "code",    "task": "ce",  "hf": "sahil2801/CodeAlpaca-20k", "split": "train",
     "system": "You are an expert programmer.",
     "map": lambda r: (r.get("instruction", ""), r.get("output", ""))},
    {"name": "math",    "task": "ce",  "hf": "meta-math/MetaMathQA", "split": "train",
     "system": "You are a math tutor.",
     "map": lambda r: (r.get("query", ""), r.get("response", ""))},
    {"name": "medical", "task": "ce",  "hf": "lavita/ChatDoctor-HealthCareMagic-100k",
     "split": "train", "system": "You are a medical expert.",
     "map": lambda r: (r.get("instruction", ""), r.get("output", ""))},
    {"name": "summary", "task": "ce",  "hf": "knkarthick/dialogsum", "split": "train",
     "system": "You are a summarization assistant.",
     "map": lambda r: ("Summarize this conversation:" + NL + r.get("dialogue", ""),
                       r.get("summary", ""))},
    {"name": "polite",  "task": "dpo", "source": "synthetic", "layers": 1, "max_len": 256,
     "system": "You are a helpful assistant."},
]

# ── where behaviors live ─────────────────────────────────────────────────────
# One repo, one folder per base model, because a behavior only loads onto the
# base it was trained against.
HF_USER       = "pfekin"
BEHAVIOR_REPO = f"{HF_USER}/lara-behaviors"
MODEL_SLUG    = BASE.split("/")[-1].lower()

# ── sizes ────────────────────────────────────────────────────────────────────
BF16   = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DTYPE  = torch.bfloat16 if BF16 else torch.float16
GAMMAS = (0.0, 0.5, 1.0, 1.5)     # 0.0 is the untouched base; past 1.0 to find the peak

MAX_LEN, N_TRAIN, N_EVAL = 384, 700, 96
EVAL_BS                  = 8      # lower it if evaluation runs out of memory
CE_STEPS, CE_LR          = 700, 2e-4
DPO_PAIRS, DPO_EVAL      = 4000, 600
DPO_STEPS, DPO_LR        = 2400, 5e-5
LAYERS, RANK, ALPHA      = 6, 128, 128
LENGTH_NORM = True
# Beta must match the scale of the logprobs. Summed logprobs differ by tens of
# nats, so 0.1 fits. Length-normalised ones differ by hundredths, and the same
# beta would leave the margin at zero.
BETA        = 2.0 if LENGTH_NORM else 0.1
NLL_LAMBDA  = 0.2     # anchors the chosen answer so the margin is not widened
                      # by pushing down on tokens both answers share

if torch.cuda.is_available():
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"{torch.cuda.get_device_name(0)}  {gb:.0f} GB  bf16={'yes' if BF16 else 'no'}")
else:
    print("no GPU: pick a GPU runtime")
print(f"base   {BASE}")
print(f"repo   {BEHAVIOR_REPO}/{MODEL_SLUG}")
print(f"{len(BEHAVIORS)} behaviors: {[b['name'] for b in BEHAVIORS]}")

NVIDIA L4  24 GB  bf16=yes
base   Qwen/Qwen3-1.7B
repo   pfekin/lara-behaviors/qwen3-1.7b
5 behaviors: ['code', 'math', 'medical', 'summary', 'polite']


## 1. What the weights look like

Not required, but it tells you how the base is stored, which decides the
footprint numbers later. Nothing in the training path depends on it: the
correction is computed in floating point whatever the weights are.

In [3]:
from huggingface_hub import hf_hub_download, list_repo_files
from safetensors import safe_open

def weight_family(repo):
    """Low-bit models are often published as their small weights written into a
    wider container. One bit per weight means a block of 128 holds two distinct
    values; ternary holds three; an ordinary model holds 128."""
    shards = sorted(f for f in list_repo_files(repo) if f.endswith(".safetensors"))
    if not shards:
        return "unknown", 16.0
    # torch rather than numpy: numpy has no bfloat16, which many models ship in.
    with safe_open(hf_hub_download(repo, shards[0]), framework="pt") as f:
        key = next((k for k in f.keys() if k.endswith("gate_proj.weight")), None)
        if key is None:
            return "unknown", 16.0
        row = f.get_tensor(key)[0].float().numpy()
    counts = {len(np.unique(row[i*128:(i+1)*128])) for i in range(16)}
    zeros = float((row[:2048] == 0.0).mean())
    if counts <= {1, 2} and zeros == 0:
        return "1-bit", 1.125
    if counts <= {1, 2, 3} and zeros > 0:
        return "ternary", 2.0
    return "full precision", 16.0


FAMILY, BITS = weight_family(BASE)
print(f"{BASE}: {FAMILY}  (~{BITS} bits per weight as shipped)")

model-00001-of-00002.safetensors: reconstructing file:   0%|          |  0.00B / 3.44GB            

model-00001-of-00002.safetensors: downloading bytes:           |  0.00B            

Qwen/Qwen3-1.7B: full precision  (~16.0 bits per weight as shipped)


## 2. Data

Each behavior gets its own dataset, streamed so nothing large is downloaded.
The preference behavior uses generated pairs instead, described in the cell.

In [4]:
from datasets import load_dataset
import itertools

def chatml(tok, system, instr, answer):
    msgs = ([{"role": "system", "content": system}] if system else []) \
         + [{"role": "user", "content": instr}]
    try:
        p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)
    except TypeError:
        p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return p, p + answer + tok.eos_token


def load_ce(tok, spec, n_train, n_eval):
    """Streamed, so nothing large is downloaded. Eval is held out from train."""
    rows = list(itertools.islice(
        load_dataset(spec["hf"], split=spec["split"], streaming=True), n_train + n_eval))
    out = []
    for r in rows:
        try:
            instr, ans = spec["map"](r)
        except Exception:
            continue
        if instr and ans and len(str(ans)) > 20:
            out.append(chatml(tok, spec["system"], str(instr)[:1500], str(ans)[:2000]))
    random.Random(0).shuffle(out)
    return out[n_eval:], out[:n_eval]


# ── synthetic preference pairs ───────────────────────────────────────────────
# Same content, same length, differing only in manner: one answer commits, the
# other hedges. Matching the length matters. If the hedged side ran longer, a
# model could score perfectly by counting words and never read the hedging.
TOPIC = ["how a compiler works", "why the sky is blue", "how to boil an egg",
         "what a database index does", "how a bicycle gear works", "why bread rises",
         "how a fuse protects a circuit", "what a checksum is for", "how sound travels",
         "why ice floats", "how a heat pump moves heat", "why metals conduct electricity",
         "what a cache miss costs", "how yeast ferments sugar",
         "why bridges have expansion joints", "how noise cancelling works",
         "what a hash function guarantees", "how sails work upwind",
         "why batteries lose capacity", "how a microphone captures sound"]
BODY = ["it comes down to a few steps that build on each other",
        "the mechanism is simpler than it first appears",
        "two things matter, and the second follows from the first",
        "one process feeds directly into another",
        "there is a cause and an effect, and they are easy to separate",
        "a small difference at the start compounds later on",
        "the energy has to go somewhere, and that is the whole trick",
        "it is a trade between speed and accuracy",
        "the shape does most of the work, not the material",
        "timing matters more than force here"]
OPENERS = [  # (direct, hedged) -- identical word counts
    ("In short, plainly", "I think, maybe"),
    ("Briefly, and clearly", "Possibly, though unsure"),
    ("Put simply, definitely", "I guess, perhaps"),
    ("The short answer is", "It might just be"),
    ("Essentially, quite clearly", "Arguably, though possibly"),
    ("At its core, certainly", "I am unsure, but"),
    ("Simply put, without doubt", "It could be perhaps"),
    ("Fundamentally, and firmly", "Conceivably, though tentatively"),
    ("Clearly, and directly", "Maybe, though doubtfully"),
    ("Precisely, and simply", "Seemingly, though vaguely"),
]
CLOSERS = [  # (confident, waffling) -- identical word counts
    ("That is the core of it.", "Though I am not really certain."),
    ("That covers the main idea.", "But do check somewhere else."),
    ("That is what makes it work.", "At least, I believe so mostly."),
    ("That is the essential part.", "Or something close to it."),
    ("That is the whole mechanism.", "More or less, I suppose."),
    ("That is the key point.", "Perhaps, though I forget now."),
    ("That explains the behaviour.", "Roughly, if memory serves."),
    ("That is the short version.", "Or thereabouts, I would gather."),
    ("That is all it amounts to.", "Though the sources may well differ."),
    ("That is the underlying reason.", "Assuming I recall this correctly."),
]
assert all(len(a.split()) == len(b.split()) for a, b in OPENERS + CLOSERS), \
    "openers and closers must match word for word, or length leaks into the signal"


def synthetic_pairs(tok, spec, n, held_out=False):
    """Training uses the first 70% of each phrase list, evaluation the rest.
    Without that split the eval pairs reuse phrasings seen thousands of times,
    every margin clears zero by a mile, and accuracy measures nothing."""
    def split(xs):
        k = int(len(xs) * 0.7)
        return xs[k:] if held_out else xs[:k]

    op, cl, tp, bd = split(OPENERS), split(CLOSERS), split(TOPIC), split(BODY)
    rng = random.Random(23 if held_out else 11)
    out = []
    for _ in range(n):
        b = rng.choice(bd)
        od, oh = rng.choice(op)
        cd, ch = rng.choice(cl)
        pr, fc = chatml(tok, spec["system"], f"Explain {rng.choice(tp)}.", f"{od} {b}. {cd}")
        _,  fj = chatml(tok, spec["system"], f"Explain {rng.choice(tp)}.", f"{oh} {b}. {ch}")
        out.append((pr, fc, fj))
    return out


_tok = AutoTokenizer.from_pretrained(BASE)
_tok.pad_token = _tok.pad_token or _tok.eos_token
DATA = {}
for b in BEHAVIORS:
    if b["task"] == "ce":
        tr, ev = load_ce(_tok, b, N_TRAIN, N_EVAL)
        DATA[b["name"]] = {"train": tr, "eval": ev}
        print(f"  {b['name']:<9} {len(tr)} train / {len(ev)} eval")
    else:
        tr = synthetic_pairs(_tok, b, DPO_PAIRS)
        ev = synthetic_pairs(_tok, b, DPO_EVAL, held_out=True)
        DATA[b["name"]] = {"pairs": tr, "pairs_eval": ev}
        se = 1.96 * (0.25 / len(ev)) ** 0.5
        shorter = np.mean([len(c.split()) < len(j.split()) for _, c, j in tr])
        print(f"  {b['name']:<9} {len(tr)} pairs / {len(ev)} eval pairs (+/-{se:.3f} at 95%)")
        print(f"  {'':<9} 'shorter is better' scores {shorter:.3f}; 0.5 means length "
              f"carries no signal")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

  math      699 train / 96 eval


README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

  medical   695 train / 96 eval


README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

  summary   700 train / 96 eval
  polite    4000 pairs / 600 eval pairs (+/-0.040 at 95%)
            'shorter is better' scores 0.346; 0.5 means length carries no signal


## 3. Measurement

In [5]:
def load_base():
    tok = AutoTokenizer.from_pretrained(BASE, clean_up_tokenization_spaces=False)
    tok.pad_token = tok.pad_token or tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(BASE, dtype=DTYPE, device_map="auto")
    return m, tok


def logp(model, ids, n_prompt, norm):
    """Single-sequence version, used by the training loop where a gradient is
    needed. Evaluation uses score_batch instead."""
    lg = model(ids).logits[:, :-1].float()
    lp = torch.log_softmax(lg, -1).gather(-1, ids[:, 1:].unsqueeze(-1)).squeeze(-1)
    lp = lp[:, n_prompt - 1:]
    return (lp.mean() if norm else lp.sum()), -lp.mean()


def enc_pair(tok, prompt, full, max_len=None):
    p = tok(prompt, add_special_tokens=False).input_ids
    f = tok(full, add_special_tokens=False).input_ids[:(max_len or MAX_LEN)]
    return torch.tensor([f]), min(len(p), len(f))


@torch.no_grad()
def score_batch(model, tok, items, max_len=None, bs=None):
    """Completion log probabilities for a list of (prompt, full) pairs.

    Batched. One forward pass per example is the difference between a minute and
    a quarter of an hour once you sweep several strengths."""
    max_len, bs = max_len or MAX_LEN, bs or EVAL_BS
    pad = tok.pad_token_id
    out = []
    for i in range(0, len(items), bs):
        chunk = items[i:i + bs]
        enc = [tok(f, add_special_tokens=False).input_ids[:max_len] for _, f in chunk]
        npr = [min(len(tok(p, add_special_tokens=False).input_ids), len(e) - 1)
               for (p, _), e in zip(chunk, enc)]
        width = max(len(e) for e in enc)
        ids = torch.full((len(enc), width), pad, dtype=torch.long)
        att = torch.zeros((len(enc), width), dtype=torch.long)
        for k, e in enumerate(enc):
            ids[k, :len(e)] = torch.tensor(e)
            att[k, :len(e)] = 1
        ids, att = ids.to(model.device), att.to(model.device)
        lg = model(ids, attention_mask=att).logits[:, :-1].float()
        lp = torch.log_softmax(lg, -1).gather(-1, ids[:, 1:].unsqueeze(-1)).squeeze(-1)
        for k, e in enumerate(enc):
            seg = lp[k, max(npr[k] - 1, 0):len(e) - 1]
            out.append((seg.sum().item(), max(seg.numel(), 1)))
    return out


def ppl(model, tok, texts):
    tot = score_batch(model, tok, texts)
    return math.exp(sum(-s for s, _ in tot) / sum(n for _, n in tot))


def ref_logprobs(model, tok, pairs, max_len=None):
    """The reference model is the base. The modules start at zero, so the model
    with nothing attached IS the reference: no second copy is needed."""
    ch = score_batch(model, tok, [(p, c) for p, c, _ in pairs], max_len)
    rj = score_batch(model, tok, [(p, j) for p, _, j in pairs], max_len)
    norm = (lambda s, n: s / n) if LENGTH_NORM else (lambda s, n: s)
    return [(norm(a, na), norm(b, nb)) for (a, na), (b, nb) in zip(ch, rj)]


def reward(model, tok, pairs, ref, max_len=None):
    """Accuracy thresholds the margin at zero, so it saturates once the
    separation is wide. The margin keeps moving after that, so both are shown."""
    ch = score_batch(model, tok, [(p, c) for p, c, _ in pairs], max_len)
    rj = score_batch(model, tok, [(p, j) for p, _, j in pairs], max_len)
    norm = (lambda s, n: s / n) if LENGTH_NORM else (lambda s, n: s)
    hit, ms = 0.0, []
    for (a, na), (b, nb), (rc, rjj) in zip(ch, rj, ref):
        d = (norm(a, na) - rc) - (norm(b, nb) - rjj)
        if not math.isfinite(d):
            continue
        # At strength 0 the policy IS the reference, so every margin is exactly
        # zero. Counting ties as a half puts that row at chance, where it belongs.
        hit += 1.0 if d > 0 else (0.5 if d == 0 else 0.0)
        ms.append(d)
    n = max(len(ms), 1)
    return hit / n, float(np.mean(ms)) if ms else float("nan")


def measure(model, tok, spec, refs):
    """One number per behavior, in whichever metric fits its objective."""
    if spec["task"] == "ce":
        return {"metric": "PPL", "want": "lower",
                "value": ppl(model, tok, DATA[spec["name"]]["eval"])}
    acc, mar = reward(model, tok, DATA[spec["name"]]["pairs_eval"],
                      refs[spec["name"]], spec.get("max_len"))
    return {"metric": "REWARD", "want": "higher", "value": acc, "margin": mar}


def eval_all(model, tok, refs, gammas, bank=None):
    """Every behavior at every strength, on one loaded model.

    bank=None measures the bare base. Otherwise each behavior is pinned in turn,
    so the model is loaded once rather than once per behavior."""
    out = {}
    for b in BEHAVIORS:
        row = {}
        for g in gammas:
            if bank is None:
                row[g] = measure(model, tok, b, refs)
            else:
                with bank.pin({b["name"]: g}):
                    row[g] = measure(model, tok, b, refs)
        out[b["name"]] = row
        print(f"  {b['name']:<9} " + "  ".join(f"{g}: {row[g]['value']:.3f}"
                                               for g in gammas))
    return out


## 4. Training

Three lines are LARA's and are marked. The rest is ordinary Hugging Face
training: the modules attach to the frozen model and everything else is frozen,
so any trainer picks up the right parameters.

Cross-entropy trains on the answer only. Labelling the prompt too would spend
gradient on predicting the question.

The preference loss carries an anchor on the chosen answer. Without it the
margin can be widened by pushing down on the tokens both answers share, which
damages the formatting they have in common.

In [6]:
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments


def train_ce(spec):
    model, tok = load_base()
    model.config.use_cache = False

    # ── LARA 1 of 3: attach. The base is frozen from here on.
    lara = LARA(model, layers=spec.get("layers", LAYERS),
                rank=spec.get("rank", RANK), alpha=spec.get("alpha", ALPHA))
    print(f"   {lara.num_trainable():,} trainable")

    rows = []
    for prompt, full in DATA[spec["name"]]["train"]:
        p = tok(prompt, add_special_tokens=False).input_ids
        f = tok(full, add_special_tokens=False).input_ids[:MAX_LEN]
        k = min(len(p), len(f))
        rows.append({"input_ids": f, "labels": [-100] * k + f[k:]})

    Trainer(model=model,
            args=TrainingArguments(
                output_dir=f"runs/{spec['name']}", max_steps=CE_STEPS,
                learning_rate=CE_LR, per_device_train_batch_size=1,
                gradient_accumulation_steps=4, gradient_checkpointing=True,
                logging_steps=350, save_strategy="no", bf16=BF16, fp16=not BF16,
                report_to=[]),
            train_dataset=Dataset.from_list(rows),
            data_collator=DataCollatorForSeq2Seq(tok, label_pad_token_id=-100)).train()

    # ── LARA 2 of 3: save. Route samples are the text the behavior produces,
    #    because that is what a router reads.
    lara.save(f"behaviors/{spec['name']}",
              route_samples=[t for _, t in DATA[spec["name"]]["train"][:200]], method="ce")
    lara.detach(); del model
    gc.collect(); torch.cuda.empty_cache()


def train_dpo(spec):
    model, tok = load_base()
    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    # ── LARA 1 of 3
    lara = LARA(model, layers=spec.get("layers", LAYERS),
                rank=spec.get("rank", RANK), alpha=spec.get("alpha", ALPHA))
    params = [p for p in model.parameters() if p.requires_grad]
    print(f"   {sum(p.numel() for p in params):,} trainable")

    pairs, ml = DATA[spec["name"]]["pairs"], spec.get("max_len", MAX_LEN)
    enc = [(enc_pair(tok, p, c, ml), enc_pair(tok, p, j, ml)) for p, c, j in pairs]

    lara.gamma = 0.0                      # strength 0 is the base: the reference
    with torch.no_grad():
        ref = [(logp(model, ic.to(model.device), nc, LENGTH_NORM)[0].item(),
                logp(model, ij.to(model.device), nj, LENGTH_NORM)[0].item())
               for (ic, nc), (ij, nj) in enc]

    lara.gamma = 1.0
    opt = torch.optim.AdamW(params, lr=DPO_LR)
    # fp16 gradients need loss scaling. HF Trainer does this for the CE runs;
    # a hand-written loop has to ask for it or the gradients go to nan.
    scaler = torch.amp.GradScaler("cuda", enabled=(DTYPE == torch.float16))
    order = list(range(len(enc))); random.Random(1).shuffle(order)
    step, run, seen, skipped = 0, 0.0, 0, 0
    while step < DPO_STEPS:
        for j in order:
            (ic, nc), (ij, nj) = enc[j]
            lc, nll = logp(model, ic.to(model.device), nc, LENGTH_NORM)
            lj, _   = logp(model, ij.to(model.device), nj, LENGTH_NORM)
            rc, rj = ref[j]
            pref = -F.logsigmoid(BETA * ((lc - rc) - (lj - rj)))
            loss = (pref + NLL_LAMBDA * nll) / 4
            if not torch.isfinite(loss):
                skipped += 1; opt.zero_grad(set_to_none=True); step += 1; continue
            scaler.scale(loss).backward()
            run += loss.item(); seen += 1
            if (step + 1) % 4 == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            step += 1
            if step % 400 == 0:
                print(f"    step {step}  loss {run / max(seen, 1) * 4:.4f}")
                run, seen = 0.0, 0
            if step >= DPO_STEPS:
                break
    if skipped:
        print(f"    {skipped} of {DPO_STEPS} batches were not finite and were skipped")
    bad = [n for n, p in model.named_parameters()
           if p.requires_grad and not torch.isfinite(p).all()]
    assert not bad, f"training diverged: {len(bad)} tensors are not finite"

    # ── LARA 2 of 3
    lara.save(f"behaviors/{spec['name']}",
              route_samples=[c for _, c, _ in pairs[:200]], method="dpo")
    lara.detach(); del model, enc
    gc.collect(); torch.cuda.empty_cache()


for b in BEHAVIORS:
    print(f"── {b['name']} ({b['task']}) ──")
    (train_ce if b["task"] == "ce" else train_dpo)(b)
    print(f"   saved behaviors/{b['name']}")

── code (ce) ──


model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

...

   3,182,592 trainable


Step,Training Loss
350,1.105175
700,0.837798


   saved behaviors/summary
── polite (dpo) ──


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

   530,432 trainable
    step 400  loss 1.1956
    step 800  loss 0.6446
    step 1200  loss 0.4491
    step 1600  loss 0.3246
    step 2000  loss 0.2321
    step 2400  loss 0.1656
   saved behaviors/polite


## 5. What each one does on its own

Loaded one at a time, at several strengths. Strength 0 is the check: the
correction is scaled to nothing, so the model should measure identically to the
bare base rather than approximately.

In [7]:
def solo_table(solo, baseline):
    print("PPL is perplexity: lower is better.")
    print("REWARD is preference accuracy against the base: higher is better, "
          "0.5 is chance.")
    print()
    hdr = f"{'behavior':<10}{'metric':<8}{'want':<7}{'base':>10}"
    print(hdr + "".join(f"{'g=' + str(g):>9}" for g in GAMMAS))
    print("-" * (len(hdr) + 9 * len(GAMMAS)))
    for b in BEHAVIORS:
        n, r = b["name"], solo[b["name"]]
        print(f"{n:<10}{r[GAMMAS[0]]['metric']:<8}{r[GAMMAS[0]]['want']:<7}"
              f"{baseline[n]:>10.3f}" + "".join(f"{r[g]['value']:>9.3f}" for g in GAMMAS))
        if b["task"] == "dpo":
            print(f"{'':<10}{'margin/tok':<8}{'higher':<7}{0.0:>10.3f}"
                  + "".join(f"{r[g]['margin']:>+9.3f}" for g in GAMMAS))
    print()
    for b in BEHAVIORS:
        n, r = b["name"], solo[b["name"]]
        if b["task"] == "ce":
            best = min(GAMMAS, key=lambda g: r[g]["value"])
            chg = (baseline[n] - r[best]["value"]) / baseline[n]
            print(f"  {n:<10} best at strength {best}: {chg:.0%} lower perplexity")
        else:
            # accuracy ties are common once the margin is wide; break on margin
            best = max(GAMMAS, key=lambda g: (r[g]["value"], r[g]["margin"]))
            print(f"  {n:<10} best at strength {best}: "
                  f"accuracy {baseline[n]:.3f} -> {r[best]['value']:.3f}, "
                  f"margin +0.000 -> {r[best]['margin']:+.3f} per token")
    drift = max(abs(solo[b["name"]][0.0]["value"] - baseline[b["name"]])
                / max(baseline[b["name"]], 1e-9) for b in BEHAVIORS)
    print()
    print(f"largest gap between the bare base and strength 0: {drift:.2%}")
    print("strength 0 reproduces the base exactly: the correction is scaled to nothing")

In [8]:
# One load, one bank, every behavior resident. Pinning them in turn gives the
# solo numbers without reloading the model for each.
model, tok = load_base(); model.eval()

refs = {b["name"]: ref_logprobs(model, tok, DATA[b["name"]]["pairs_eval"],
                                b.get("max_len"))
        for b in BEHAVIORS if b["task"] == "dpo"}

print("bare base, nothing attached:")
baseline = {n: r[0.0]["value"]
            for n, r in eval_all(model, tok, refs, (0.0,)).items()}

bank = Bank(model, tok)
for b in BEHAVIORS:
    bank.add(b["name"], f"behaviors/{b['name']}")
print()
print("each behavior pinned in turn:")
solo = eval_all(model, tok, refs, GAMMAS, bank=bank)

print()
solo_table(solo, baseline)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

bare base, nothing attached:
  code      0.0: 11.555
  math      0.0: 3.571
  medical   0.0: 129.816
  summary   0.0: 701.252
  polite    0.0: 0.500

each behavior pinned in turn:
  code      0.0: 11.555  0.5: 1.926  1.0: 1.833  1.5: 1.925
  math      0.0: 3.571  0.5: 1.291  1.0: 1.243  1.5: 1.270
  medical   0.0: 129.816  0.5: 16.940  1.0: 14.507  1.5: 16.304
  summary   0.0: 701.252  0.5: 3.267  1.0: 2.762  1.5: 3.085
  polite    0.0: 0.500  0.5: 0.993  1.0: 1.000  1.5: 1.000

PPL is perplexity: lower is better.
REWARD is preference accuracy against the base: higher is better, 0.5 is chance.

behavior  metric  want         base    g=0.0    g=0.5    g=1.0    g=1.5
-----------------------------------------------------------------------
code      PPL     lower      11.555   11.555    1.926    1.833    1.925
math      PPL     lower       3.571    3.571    1.291    1.243    1.270
medical   PPL     lower     129.816  129.816   16.940   14.507   16.304
summary   PPL     lower     701.252  7

## 6. Publish

One repo, one folder per base model, because a behavior only loads onto the base
it was trained against. `routed_load` reads the same two constants and finds
them.

A write token is read from the environment, from `HF_TOKEN` in Colab's secrets
panel, or asked for once.

In [9]:
UPLOAD = False

if UPLOAD:
    from huggingface_hub import HfApi, get_token, login

    if get_token() is None:
        login()
    api = HfApi()
    print("uploading as", api.whoami()["name"])     # fails early if the token is read-only
    api.create_repo(BEHAVIOR_REPO, repo_type="model", exist_ok=True)

    card = ["---", "library_name: lara", "license: apache-2.0",
            "tags:", "  - lara", "  - adapter", "  - behavior",
            f"base_model: {BASE}", "---", "",
            f"# LARA behaviors for `{BASE}`", "",
            f"Base: `{BASE}` ({FAMILY}). Each folder is a behavior: a low-rank",
            "correction applied between transformer blocks. The base is never",
            "modified, and strength 0 reproduces it exactly.", "",
            "| behavior | objective | modules | size |", "|---|---|---|---|"]
    for b in BEHAVIORS:
        mb = sum(os.path.getsize(os.path.join(d, f))
                 for d, _, fs in os.walk(f"behaviors/{b['name']}") for f in fs) / 1e6
        card.append(f"| `{b['name']}` | {b['task'].upper()} | "
                    f"{b.get('layers', LAYERS)} | {mb:.1f} MB |")
    card += ["", "```python", "from lara import Bank",
             "bank = Bank(model, tok)", f'bank.add("{BEHAVIORS[0]["name"]}", '
             f'"{MODEL_SLUG}/{BEHAVIORS[0]["name"]}")', "```", "",
             "https://github.com/pfekin/LARA · https://arxiv.org/abs/2607.28669"]
    os.makedirs("behaviors", exist_ok=True)
    open("behaviors/README.md", "w").write(NL.join(card))

    # Replaces this model's folder and leaves other models in the repo alone.
    api.upload_folder(folder_path="behaviors", repo_id=BEHAVIOR_REPO,
                      path_in_repo=MODEL_SLUG, repo_type="model",
                      commit_message=f"behaviors for {BASE}",
                      delete_patterns=[f"{MODEL_SLUG}/*"])
    print(f"pushed to https://huggingface.co/{BEHAVIOR_REPO}/tree/main/{MODEL_SLUG}")
    print("now run routed_load with the same BASE")

uploading as pfekin
pushed to https://huggingface.co/pfekin/lara-behaviors/tree/main/qwen3-1.7b
now run routed_load with the same BASE


## Notes

Strength 0 reproducing the base is a property of adding a scaled correction to
weights that were never touched, not an approximation that happens to be close.

The preference behavior uses a single module where the fine-tuned ones use six.
Preference training is much less sensitive to placement, and the resulting file
is a fraction of the size.

If a behavior trains but does not move its metric, the two useful questions are
whether it fitted its own training data at all, and whether the evaluation set
shares phrasing with the training set. Both are easier to answer than tuning.

- https://github.com/pfekin/LARA
- https://arxiv.org/abs/2607.28669